In [9]:
import json
import re

from sympy.integrals.meijerint_doc import category

In [1]:
def load_data(filepath):
    """Hàm này đọc dữ liệu từ file JSON."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return data
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {filepath}.")
        return {}
    except json.JSONDecodeError:
        print(f"Lỗi: File {filepath} bị hỏng hoặc trống.")
        return {}
def save_data(filepath, data):
    """Hàm này ghi đè dữ liệu (data) vào file JSON."""
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        return True
    except Exception as e:
        print(f"Đã xảy ra lỗi khi lưu file: {e}")
        return False

In [2]:
def add_entry_abbreviation(filepath, key, value):
    category ="abbreviations"
    """Thêm một cặp key-value mới (đảm bảo key là chữ thường)."""
    print(f"Đang thêm: {key} -> {value} vào mục {category}...")
    data = load_data(filepath)

    if category not in data:
        data[category] = {}

    # Luôn lưu key viết tắt ở dạng chữ thường
    data[category][key.lower()] = value

    if save_data(filepath, data):
        print("Thêm thành công và đã lưu file!")
    else:
        print("Không thể lưu file.")

In [3]:
def add_entry_punctuation(filepath, key, value):
    category ="punctuation"
    """Thêm một cặp key-value mới (đảm bảo key là chữ thường)."""
    print(f"Đang thêm: {key} -> {value} vào mục {category}...")
    data = load_data(filepath)

    if category not in data:
        data[category] = {}

    # Luôn lưu key viết tắt ở dạng chữ thường
    data[category][key.lower()] = value

    if save_data(filepath, data):
        print("Thêm thành công và đã lưu file!")
    else:
        print("Không thể lưu file.")

In [5]:
def process_sentence_regex(sentence, filepath):
    """
    Xử lý câu bằng Regex để thay thế chính xác và hiệu quả.
    """

    data = load_data(filepath)
    if not data:
        return "Lỗi: Không thể tải từ điển."

    processed_sentence = sentence

    # --- Bước 1: Xử lý Từ viết tắt (abbreviations) ---
    # Chúng ta cần ranh giới từ \b và không phân biệt hoa thường

    abbreviations = data.get('abbreviations', {})
    if abbreviations:
        # Sắp xếp các key: ưu tiên key dài nhất trước
        # Rất quan trọng: để "ntn" được thay trước "n"
        sorted_keys = sorted(abbreviations.keys(), key=len, reverse=True)

        # Tạo 1 mẫu Regex duy nhất, ví dụ: r'\b(ntn|ko|k|dc|sp|ad|shop)\b'
        # re.escape() để đảm bảo key an toàn (ví dụ key là "a+b")
        pattern_abbr = r"\b(" + r"|".join(re.escape(k) for k in sorted_keys) + r")\b"

        # re.sub có thể nhận một hàm (lambda) để quyết định thay thế
        # m.group(1) là key đã tìm thấy (ví dụ: "ad", "Ko", "SP")
        # m.group(1).lower() để tra cứu trong dict (vì dict của ta là chữ thường)
        processed_sentence = re.sub(
            pattern_abbr,
            # m.group(1) là nội dung trong cặp ngoặc đơn ()
            lambda m: abbreviations[m.group(1).lower()],
            processed_sentence,
            flags=re.IGNORECASE  # Không phân biệt hoa thường
        )

    # --- Bước 2: Xử lý Dấu câu (punctuation) ---
    # Chúng ta KHÔNG dùng \b và CÓ phân biệt hoa thường

    punctuation = data.get('punctuation', {})
    if punctuation:
        # Sắp xếp (dù ít quan trọng hơn nhưng là thói quen tốt)
        sorted_keys = sorted(punctuation.keys(), key=len, reverse=True)

        # Tạo mẫu: r'(\?|\!|\.)'
        # Không có \b
        pattern_punct = r"(" + r"|".join(re.escape(k) for k in sorted_keys) + r")"

        processed_sentence = re.sub(
            pattern_punct,
            lambda m: punctuation[m.group(1)],
            processed_sentence
            # Không có flags=re.IGNORECASE
        )

    return processed_sentence

In [7]:
%%writefile dictionary.json
{
    "abbreviations": {
        "ko": "không",
        "k": "không",
        "dc": "được",
        "sp": "sản phẩm",
        "ad": "admin",
        "shop": "cửa hàng",
        "ntn": "như thế nào"
    },
    "punctuation": {
        "?": " [DẤU_HỎI] ",
        "!": " [DẤU_CHẤM_THAN] ",
        ".": " [DẤU_CHẤM] "
    }
}

Overwriting dictionary.json


In [10]:
filepath = 'dictionary.json'

# Thêm từ 'bt' (bình thường) vào file
add_entry_abbreviation(filepath, 'bt', 'bình thường')

print("\n--- KẾT QUẢ TEST ---")

# Ví dụ 1: Giải quyết vấn đề 'bad request'
test_1 = "ad ơi, sp này dc ko? bad request."
print(f"Gốc: {test_1}")
print(f"Sửa: {process_sentence_regex(test_1, filepath)}\n")

# Ví dụ 2: Test không phân biệt hoa thường (IGNORECASE)
test_2 = "AD ơi, SP này thế nào?"
print(f"Gốc: {test_2}")
print(f"Sửa: {process_sentence_regex(test_2, filepath)}\n")

# Ví dụ 3: Test key dài nhất (ntn vs n)
test_3 = "hàng ntn vậy shop?"
print(f"Gốc: {test_3}")
print(f"Sửa: {process_sentence_regex(test_3, filepath)}\n")

# Ví dụ 4: Test key "bt" mới thêm
test_4 = "mọi thứ bt."
print(f"Gốc: {test_4}")
print(f"Sửa: {process_sentence_regex(test_4, filepath)}\n")

Đang thêm: bt -> bình thường vào mục abbreviations...
Thêm thành công và đã lưu file!

--- KẾT QUẢ TEST ---
Gốc: ad ơi, sp này dc ko? bad request.
Sửa: admin ơi, sản phẩm này được không [DẤU_HỎI]  bad request [DẤU_CHẤM] 

Gốc: AD ơi, SP này thế nào?
Sửa: admin ơi, sản phẩm này thế nào [DẤU_HỎI] 

Gốc: hàng ntn vậy shop?
Sửa: hàng như thế nào vậy cửa hàng [DẤU_HỎI] 

Gốc: mọi thứ bt.
Sửa: mọi thứ bình thường [DẤU_CHẤM] 

